In [ ]:
!pip install -U ultralytics roboflow pyyaml pandas

In [ ]:
# =========================
# Project Config
# =========================
# Roboflow 프로젝트 정보
ROBOFLOW_WORKSPACE = "2026-oss"
ROBOFLOW_PROJECT = "ai-picture-book-object-detection"

# Roboflow 최신 통합 데이터셋 버전
ROBOFLOW_VERSION = 12

# 사용할 YOLO 모델과 학습 run 이름
MODEL_NAME = "yolo11n.pt"
RUN_NAME = "picture_book_yolo11n_v12"

# 학습 설정
IMG_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 16
CONF_THRESHOLD = 0.25

In [ ]:
# Colab에서 GPU가 정상적으로 연결되었는지 확인
!nvidia-smi

In [ ]:
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

print("Roboflow API Key loaded:", ROBOFLOW_API_KEY is not None)

In [ ]:
# Roboflow에서 v12 데이터셋 다운로드
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

# "yolov8"은 모델명이 아니라 데이터셋 export format임
# 실제 학습 모델은 아래에서 YOLO("yolo11n.pt")로 지정함
dataset = project.version(12).download("yolov8")

print("Dataset downloaded to:", dataset.location)

In [ ]:
# Roboflow API Key를 Colab Secrets에서 불러오고, 없으면 직접 입력받음
from getpass import getpass

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Roboflow API Key를 입력하세요: ")

# 앞뒤 공백 제거
ROBOFLOW_API_KEY = ROBOFLOW_API_KEY.strip()

print("Roboflow API Key loaded:", ROBOFLOW_API_KEY is not None and len(ROBOFLOW_API_KEY) > 0)
print("API Key length:", len(ROBOFLOW_API_KEY))

In [ ]:
# data.yaml 파일을 확인하고, 학습 경로를 절대경로로 보정
import os
import yaml

yaml_path = os.path.join(dataset.location, "data.yaml")

print("Original data.yaml path:", yaml_path)

with open(yaml_path, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

print("Original data.yaml:")
print(data_yaml)

# 경로 오류를 줄이기 위해 train/valid/test 이미지 경로를 절대경로로 수정
data_yaml["train"] = os.path.join(dataset.location, "train", "images")
data_yaml["val"] = os.path.join(dataset.location, "valid", "images")
data_yaml["test"] = os.path.join(dataset.location, "test", "images")

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print("\nUpdated data.yaml:")
with open(yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# v12 데이터셋에 10개 클래스가 모두 포함되어 있는지 확인

EXPECTED_CLASSES = [
    "book_flower",
    "book_flowerpot",
    "book_monkey",
    "book_stone",
    "braille",
    "tactile_flower",
    "tactile_flowerpot",
    "tactile_monkey",
    "tactile_stone",
    "text",
]

with open(yaml_path, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

names = data_yaml.get("names")

# names가 dict 또는 list 형태일 수 있으므로 둘 다 처리
if isinstance(names, dict):
    actual_classes = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
else:
    actual_classes = names

print("Actual classes:")
for idx, name in enumerate(actual_classes):
    print(f"{idx}: {name}")

print("\nExpected class count:", len(EXPECTED_CLASSES))
print("Actual class count:", len(actual_classes))

missing_classes = sorted(set(EXPECTED_CLASSES) - set(actual_classes))
extra_classes = sorted(set(actual_classes) - set(EXPECTED_CLASSES))

print("\nMissing classes:", missing_classes)
print("Extra classes:", extra_classes)

assert len(actual_classes) == len(EXPECTED_CLASSES), "클래스 개수가 예상과 다릅니다."
assert set(actual_classes) == set(EXPECTED_CLASSES), "클래스 목록이 예상과 다릅니다."

print("\n클래스 검증 완료: v12 데이터셋에 10개 클래스가 모두 포함되어 있습니다.")

In [ ]:
# train, valid, test 데이터가 정상적으로 나뉘어 있는지 확인
import glob

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(dataset.location, split, "images")
    label_dir = os.path.join(dataset.location, split, "labels")

    image_count = len(glob.glob(os.path.join(image_dir, "*")))
    label_count = len(glob.glob(os.path.join(label_dir, "*")))

    print(f"{split}: images={image_count}, labels={label_count}")

In [ ]:
# YOLO11n 모델을 불러와 v12 데이터셋으로 학습
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=yaml_path,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    name=RUN_NAME,
    project="runs/detect"
)

In [ ]:
# YOLO11n v12 학습 결과의 best.pt 경로 확인
from pathlib import Path

best_weights = [
    path for path in Path("/content").rglob("best.pt")
    if RUN_NAME in str(path)
]

best_weights = sorted(best_weights)

print("best.pt files:")
for path in best_weights:
    print(path)

assert len(best_weights) > 0, "YOLO11n v12 best.pt 파일을 찾지 못했습니다."

best_model_path = str(best_weights[-1])

print("\nBest model path:", best_model_path)

In [ ]:
# best.pt 모델을 test set 기준으로 평가
best_model = YOLO(best_model_path)

metrics = best_model.val(
    data=yaml_path,
    split="test",
    imgsz=IMG_SIZE
)

In [ ]:
# 전체 성능 지표 정리
summary = {
    "model": MODEL_NAME,
    "dataset_version": ROBOFLOW_VERSION,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP@0.5": float(metrics.box.map50),
    "mAP@0.5:0.95": float(metrics.box.map),
    "best_model_path": best_model_path,
}

summary

In [ ]:
# 클래스별 mAP 성능 확인
import pandas as pd

class_result_rows = []

ap50_values = getattr(metrics.box, "ap50", None)
ap_values = getattr(metrics.box, "ap", None)

for idx, class_name in enumerate(actual_classes):
    row = {
        "class_id": idx,
        "class_name": class_name,
        "mAP@0.5": float(ap50_values[idx]) if ap50_values is not None and idx < len(ap50_values) else None,
        "mAP@0.5:0.95": float(ap_values[idx]) if ap_values is not None and idx < len(ap_values) else None,
    }
    class_result_rows.append(row)

class_results_df = pd.DataFrame(class_result_rows)
class_results_df

In [ ]:
# 평가 결과를 JSON, CSV 파일로 저장
import json

os.makedirs("results", exist_ok=True)

summary_path = "results/yolo11n_v12_summary.json"
class_result_path = "results/yolo11n_v12_class_results.csv"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

class_results_df.to_csv(class_result_path, index=False, encoding="utf-8-sig")

print("Saved summary:", summary_path)
print("Saved class results:", class_result_path)

In [ ]:
# test 이미지에 대해 실제 예측 결과 저장
test_image_path = os.path.join(dataset.location, "test", "images")

predict_results = best_model.predict(
    source=test_image_path,
    conf=CONF_THRESHOLD,
    save=True,
    name=f"{RUN_NAME}_test_predictions"
)

In [ ]:
# 저장된 예측 결과 이미지 일부 확인
from IPython.display import Image, display

result_images = sorted(glob.glob(f"runs/detect/{RUN_NAME}_test_predictions*/*.jpg"))

print("Detected images:", len(result_images))

for img_path in result_images[:10]:
    display(Image(filename=img_path))

In [ ]:
# 백엔드 연동 시 필요한 모델 정보 정리
model_info = {
    "model_name": MODEL_NAME,
    "run_name": RUN_NAME,
    "dataset_version": ROBOFLOW_VERSION,
    "input": {
        "type": "image_or_webcam_frame",
        "image_size": IMG_SIZE,
    },
    "output": {
        "class_name": "detected object class",
        "confidence": "detection confidence score",
        "bbox": "[x1, y1, x2, y2]",
    },
    "classes": actual_classes,
    "best_model_path": best_model_path,
    "metrics": summary,
}

model_info_path = "results/yolo11n_v12_model_info.json"

with open(model_info_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print("Saved model info:", model_info_path)
model_info